# TickDB Streaming - Spark Structured Streaming

In [22]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
import time
import os

In [23]:
spark = SparkSession.builder \
    .appName("TickDBStreaming") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark: {spark.version}")

Spark: 3.5.0


In [24]:
event_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("last_price", DoubleType(), True),
    StructField("volume_24h", DoubleType(), True),
    StructField("high_24h", DoubleType(), True),
    StructField("low_24h", DoubleType(), True),
    StructField("price_change_24h", DoubleType(), True),
    StructField("price_change_percent_24h", DoubleType(), True),
    StructField("timestamp", LongType(), True),
    StructField("event_timestamp", LongType(), True),
    StructField("market", StringType(), True)
])

In [4]:
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "tickdb-market-data") \
    .option("startingOffsets", "earliest") \
    .load()

print("✓ Conectado a Kafka")

✓ Conectado a Kafka


In [5]:
df_parsed = df_kafka.select(
    from_json(col("value").cast("string"), event_schema).alias("data"),
    col("timestamp").alias("kafka_timestamp"),
    col("topic"),
    col("partition"),
    col("offset")
).select("data.*", "kafka_timestamp", "topic", "partition", "offset")

print("✓ JSON parseado")

✓ JSON parseado


In [6]:
df_metrics = df_parsed \
    .withColumn("timestamp_ts", (col("timestamp") / 1000).cast("timestamp")) \
    .withColumn("latency_ms", col("event_timestamp") - col("timestamp")) \
    .withColumn("processing_time", current_timestamp())

print("✓ Métricas calculadas")

✓ Métricas calculadas


In [7]:
df_valid = df_metrics.filter(
    col("symbol").isNotNull() & 
    col("last_price").isNotNull()
)

print("✓ Datos filtrados")

✓ Datos filtrados


## MOSTRAR STREAM EN TIEMPO REAL

In [8]:
print("⏳ Esperando datos...")
print("=" * 80)

query = df_valid.select(
    "symbol",
    "last_price",
    "price_change_percent_24h",
    "market",
    "latency_ms"
).writeStream \
    .format("console") \
    .option("truncate", "false") \
    .trigger(processingTime="5 seconds") \
    .start()

⏳ Esperando datos...


## GUARDAR EN PARQUET (opcional)

In [9]:
output_path = "/home/jovyan/work/tickdb_parquet"
checkpoint_path = "/home/jovyan/work/checkpoint"

os.makedirs(output_path, exist_ok=True)
os.makedirs(checkpoint_path, exist_ok=True)

print(f"Output: {output_path}")

Output: /home/jovyan/work/tickdb_parquet


In [10]:
df_windowed = df_valid \
    .withWatermark("timestamp_ts", "10 minutes") \
    .groupBy(
        window("timestamp_ts", "1 minute"),
        col("market")
    ) \
    .agg(
        count("*").alias("event_count"),
        avg("last_price").alias("avg_price"),
        avg("latency_ms").alias("avg_latency_ms"),
        max("price_change_percent_24h").alias("max_change")
)

In [11]:
query_parquet = df_windowed \
    .writeStream \
    .format("parquet") \
    .option("path", output_path) \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(processingTime="10 seconds") \
    .outputMode("append") \
    .start()

print(f"✓ Guardando en: {output_path}")

✓ Guardando en: /home/jovyan/work/tickdb_parquet


In [12]:
# Ver métricas después de 30 segundos
import time
time.sleep(30)

if query.lastProgress:
    p = query.lastProgress
    print("=== MÉTRICAS ===")
    print(f"Eventos: {p.get('numInputRows', 0)}")
    print(f"Events/sec: {p.get('inputRowsPerSecond', 0):.2f}")

=== MÉTRICAS ===
Eventos: 21
Events/sec: 4.21


In [13]:
# Detener todas las queries activas
for q in spark.streams.active:
    q.stop()
print("Queries detenidas")

Queries detenidas


In [14]:
# Nueva query con output
query = df_valid.select(
    "symbol",
    "last_price", 
    "price_change_percent_24h",
    "market",
    "latency_ms"
).writeStream \
    .format("console") \
    .option("truncate", "false") \
    .trigger(processingTime="3 seconds") \
    .start()

print("Stream iniciado...")

Stream iniciado...


In [15]:
# Ver último batch procesado
if query.lastProgress:
    print("Último batch:")
    print(query.lastProgress)

In [16]:
df_parquet = spark.read.parquet("/home/jovyan/work/tickdb_parquet")
df_parquet.show(100, truncate=False)

+------------------------------------------+---------+-----------+------------------+------------------+----------+
|window                                    |market   |event_count|avg_price         |avg_latency_ms    |max_change|
+------------------------------------------+---------+-----------+------------------+------------------+----------+
|{2026-05-19 16:18:00, 2026-05-19 16:19:00}|forex    |16         |1165.439890625    |1383.4375         |0.09      |
|{2026-05-19 17:17:00, 2026-05-19 17:18:00}|crypto   |17         |18784.69411764706 |780.9411764705883 |0.82      |
|{2026-05-19 15:40:00, 2026-05-19 15:41:00}|us_stocks|28         |369.38614285714283|805.0357142857143 |-0.14     |
|{2026-05-19 22:33:00, 2026-05-19 22:34:00}|forex    |37         |1129.2617686486487|2389.0            |0.14      |
|{2026-05-19 17:22:00, 2026-05-19 17:23:00}|forex    |20         |1167.1422099999997|1545.7            |-0.02     |
|{2026-05-19 22:42:00, 2026-05-19 22:43:00}|crypto   |36         |19896.

In [17]:
df_parquet = spark.read.parquet("/home/jovyan/work/tickdb_parquet")
df_parquet.show(10, truncate=False)

+------------------------------------------+---------+-----------+------------------+------------------+----------+
|window                                    |market   |event_count|avg_price         |avg_latency_ms    |max_change|
+------------------------------------------+---------+-----------+------------------+------------------+----------+
|{2026-05-19 16:18:00, 2026-05-19 16:19:00}|forex    |16         |1165.439890625    |1383.4375         |0.09      |
|{2026-05-19 17:17:00, 2026-05-19 17:18:00}|crypto   |17         |18784.69411764706 |780.9411764705883 |0.82      |
|{2026-05-19 15:40:00, 2026-05-19 15:41:00}|us_stocks|28         |369.38614285714283|805.0357142857143 |-0.14     |
|{2026-05-19 22:33:00, 2026-05-19 22:34:00}|forex    |37         |1129.2617686486487|2389.0            |0.14      |
|{2026-05-19 17:22:00, 2026-05-19 17:23:00}|forex    |20         |1167.1422099999997|1545.7            |-0.02     |
|{2026-05-19 22:42:00, 2026-05-19 22:43:00}|crypto   |36         |19896.

In [18]:
df_parquet = spark.read.parquet("/home/jovyan/work/tickdb_parquet")
df_parquet.show(10, truncate=False)

+------------------------------------------+---------+-----------+------------------+------------------+----------+
|window                                    |market   |event_count|avg_price         |avg_latency_ms    |max_change|
+------------------------------------------+---------+-----------+------------------+------------------+----------+
|{2026-05-19 16:18:00, 2026-05-19 16:19:00}|forex    |16         |1165.439890625    |1383.4375         |0.09      |
|{2026-05-19 17:17:00, 2026-05-19 17:18:00}|crypto   |17         |18784.69411764706 |780.9411764705883 |0.82      |
|{2026-05-19 15:40:00, 2026-05-19 15:41:00}|us_stocks|28         |369.38614285714283|805.0357142857143 |-0.14     |
|{2026-05-19 22:33:00, 2026-05-19 22:34:00}|forex    |37         |1129.2617686486487|2389.0            |0.14      |
|{2026-05-19 17:22:00, 2026-05-19 17:23:00}|forex    |20         |1167.1422099999997|1545.7            |-0.02     |
|{2026-05-19 22:42:00, 2026-05-19 22:43:00}|crypto   |36         |19896.

In [19]:
query_parquet = df_windowed \
    .writeStream \
    .queryName("parquet_sink") \
    .format("parquet") \
    .option("path", "/home/jovyan/work/tickdb_parquet") \
    .option("checkpointLocation", "/home/jovyan/work/checkpoint") \
    .trigger(processingTime="10 seconds") \
    .outputMode("append") \
    .start()

print("✓ Guardando en: /home/jovyan/work/tickdb_parquet")

✓ Guardando en: /home/jovyan/work/tickdb_parquet


In [20]:
print("=== PARÁMETROS ===")
print("Trigger: 5 segundos")
print("Watermark: 10 minutes")
print("Ventana: 1 minute")
print("Output Mode: append")
print("Checkpoint: /home/jovyan/work/checkpoint")

=== PARÁMETROS ===
Trigger: 5 segundos
Watermark: 10 minutes
Ventana: 1 minute
Output Mode: append
Checkpoint: /home/jovyan/work/checkpoint
